# job_classify via SDK — ViT-base/16 on Food-101

Mirrors `../job_classify/` — federated fine-tuning of `nateraw/vit-base-food101` (101 classes) at 224×224.

## Recipe (matches `job_classify/configs/`)

| Knob | Value |
|---|---|
| `pretrained_id` | `nateraw/vit-base-food101` (timm-loadable equivalent: `vit_base_patch16_224.augreg2_in21k_ft_in1k`) |
| `num_classes` | 101 |
| `learning_rate` | 2e-4 |
| `batch_size` | 128 |
| `local_epochs` | 1 |
| `num_rounds` | 5 |

In [ ]:
# ── Connect to ResonTech ─────────────────────────────────────────────────────
# Reads the platform api key + S3 keys from notebooks/.env (see .env.example).
# pip install -U "resontech>=0.2.1" python-dotenv   # Python 3.11+
import os

from dotenv import load_dotenv
load_dotenv()

from resontech import (
    ResonTech, ResonTechConfig,
    TrainingConfig, FederationConfig, ModelConfig,
)

_kwargs = dict(
    base_url=os.environ.get("RESON_BASE_URL",  "https://api.beta.reson.tech"),
    platform_api_key=os.environ["RESON_API_KEY"],
    s3_access_key_id=os.environ["RESON_S3_KEY"],
    s3_secret_access_key=os.environ["RESON_S3_SECRET"],
)
# Optional URL overrides (for beta / self-hosted deployments)
if "RESON_S3_ENDPOINT" in os.environ:
    _kwargs["s3_endpoint"] = os.environ["RESON_S3_ENDPOINT"]
if "RESON_DASHBOARD_URL" in os.environ:
    _kwargs["dashboard_url"] = os.environ["RESON_DASHBOARD_URL"]
if "RESON_S3_REGION" in os.environ:
    _kwargs["s3_region"] = os.environ["RESON_S3_REGION"]
if "RESON_S3_BUCKET" in os.environ:
    _kwargs["s3_bucket_alias"] = os.environ["RESON_S3_BUCKET"]

config = ResonTechConfig(**_kwargs)
sdk = ResonTech(config)   # api-key auth — no login step

In [ ]:
"""Mirrors job_classify/scripts/{model_def.py, timm_utils.py}."""
import os

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import timm


PRETRAINED_ID = "vit_base_patch16_224.augreg2_in21k_ft_in1k"
NUM_CLASSES = 101
IMG_SIZE = 224


class Model(nn.Module):
    def __init__(self, pretrained_id: str = PRETRAINED_ID, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.inner = timm.create_model(
            pretrained_id, pretrained=True, num_classes=num_classes
        )

    def forward(self, x):
        return self.inner(x)


def _make_loaders(data_root: str, imgsz: int, batch: int):
    tx = transforms.Compose([
        transforms.Resize((imgsz, imgsz)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    train_ds = datasets.ImageFolder(
        os.path.join(data_root, "images", "train"), transform=tx
    )
    return DataLoader(
        train_ds, batch_size=batch, shuffle=True, num_workers=2, pin_memory=True
    ), len(train_ds)


def fl_train(model, env, out_dir, logger=None):
    data_root = env["DATA_ROOT"]
    epochs    = int(env["EPOCHS"])
    batch     = int(env["BATCH_SIZE"])
    lr        = float(env["LR"])
    imgsz     = int(env.get("IMG_SIZE", IMG_SIZE))

    loader, n = _make_loaders(data_root, imgsz, batch)
    device = next(model.parameters()).device
    optim = torch.optim.AdamW(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    model.train()
    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optim.zero_grad(set_to_none=True)
            loss_fn(model(x), y).backward()
            optim.step()

    return {
        "weights": {k: v.detach().cpu() for k, v in model.state_dict().items()},
        "samples": n,
    }

In [ ]:
training = TrainingConfig(
    local_epochs=1,
    batch_size=128,
    learning_rate=0.0002,
    extra={"IMG_SIZE": 224},
)

federation = FederationConfig(num_rounds=5, min_clients=1)

model_config = ModelConfig(model_args={"pretrained_id": "vit_base_patch16_224.augreg2_in21k_ft_in1k", "num_classes": 101})

job = sdk.rt_submit(
    model=Model,
    name="job_classify",
    shards_dir="../job_classify/shards",
    requirements_txt="../job_classify/requirements/requirements.txt",
    training=training,
    federation=federation,
    model_config=model_config,
)

print(f"Submitted: {job.id}")
print(f"Track at:  {job.dashboard_url}")

## After submission

Open `job.dashboard_url` to follow logs and download the final checkpoint when training finishes. The platform writes the aggregated model to `result.pt` in the bucket and surfaces a download button on the job page.